In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem

# Assignment_Session 14: RDKit exercise

In session 14, we had a look at a solubility dataset. You will use that (`delaney.csv`) and a dataset related to the last assignment (`ecoli_mics_sample.csv`). The datasets have roughly the same size, but the latter needs a bit of cleaning (remove entries that do not have a SMILES or a MIC defined).

This assignment is aimed at deepening your RDKit skills and your understanding of encoding molecular structures. You will produce some functions that you should be able to reuse on both datasets!

## Tasks

1) Load the data. Pay attention to relative paths.


In [2]:
df_delaney = pd.read_csv("data/delaney.csv")
df_ecoli = pd.read_csv("data/ecoli_mics_sample.csv")

In [3]:
print(f" Duplicates: {df_delaney.duplicated().sum()}, \n ____ \n # NA : {df_delaney.isna().sum()}, \n ____\n Unique vals{df_delaney.nunique(axis=0)}")

 Duplicates: 0, 
 ____ 
 # NA : Compound ID    0
logS           0
smiles         0
dtype: int64, 
 ____
 Unique valsCompound ID    1143
logS            743
smiles         1128
dtype: int64


In [4]:
print(f" Duplicates: {df_ecoli.duplicated().sum()}, \n ____ \n # NA : {df_ecoli.isna().sum()}, \n ____\n Unique vals{df_ecoli.nunique(axis=0)}")

 Duplicates: 58, 
 ____ 
 # NA : chembl_id    0
smiles       0
mic          2
unit         0
dtype: int64, 
 ____
 Unique valschembl_id    888
smiles       888
mic          149
unit           1
dtype: int64


In [5]:
df_ecoli.drop_duplicates(inplace=True)
df_ecoli.dropna(inplace=True)

In [6]:
df_ecoli.head()

,chembl_id,smiles,mic,unit
0,CHEMBL1689756,CN(C)[C@@H]1C(O)=C(C(N)=O)C(=O)[C@@]2(O)C(O)=C...,0.25,ug.mL-1
1,CHEMBL1222951,CO/N=C1/CCN(c2c(F)cc3c(=O)c(C(=O)O)cn(C4CC4)c3...,8.00,ug.mL-1
2,CHEMBL2403434,COc1cccc(CC(=O)Nc2nc(-c3ccc(Br)cc3)cs2)c1,12.50,ug.mL-1
3,CHEMBL148,C[C@@H](O)[C@H]1C(=O)N2C(C(=O)O)=C(SCCNC=N)C[C...,0.19,ug.mL-1
4,CHEMBL22,COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC,0.25,ug.mL-1


In [7]:
df_delaney.head()

,Compound ID,logS,smiles
0,"1,1,1,2-Tetrachloroethane",-2.18,ClCC(Cl)(Cl)Cl
1,"1,1,1-Trichloroethane",-2.00,CC(Cl)(Cl)Cl
2,"1,1,2,2-Tetrachloroethane",-1.74,ClC(Cl)C(Cl)Cl
3,"1,1,2-Trichloroethane",-1.48,ClCC(Cl)Cl
4,"1,1,2-Trichlorotrifluoroethane",-3.04,FC(F)(Cl)C(F)(Cl)Cl


2) Add a column with images of your sturctures to the dataframe (Hint: Use RdKit's `PandasTools`)


In [8]:
from rdkit.Chem import PandasTools
PandasTools.AddMoleculeColumnToFrame(df_delaney,'smiles','molecule',includeFingerprints=True)
PandasTools.AddMoleculeColumnToFrame(df_ecoli,'smiles','molecule',includeFingerprints=True)

In [118]:
#PandasTools.FrameToGridImage(df_delaney, column='molecule')

In [119]:
#PandasTools.FrameToGridImage(df_ecoli, column='molecule')

3) Create your own fingerprint system: Write a function that uses SMARTS in a substructure search to assess if a certain motif (e.g. aromatic ring, ketone, carbon chain longer than 3 C, double bonds, etc) is present which you believe is important. The resulting Boolean should be saved in a list (i.e. your custom fingerprint). Look for 5-10 motifs and append this list of binary values. In the end, your fingerprint should look something like this: `[0, 0, 1, 1, 1, 0, 1, 0, 1, 0]`, with each value representing a certain molecular functional group / motif.

In [53]:
motifs = ["[CX3](=O)[OX2H]" , "c1ccccc1", "[NX3][CX3](=[OX1])[#6]", "[#6][CX3](=O)[#6]", "[CX3H1](=O)[#6]", "[OX2H]", "[NX3;H2,H1;!$(NC=O)]", "[#6][F,Cl,Br,I]"]
# CA, aromatic ring, amide, ketone, aldehyde, Hydroxyl, Primary or secondary amine, not amide, halogen


def substructures(mol):
    fp = []
    for motif in motifs:
        fp.append(mol.HasSubstructMatch(Chem.MolFromSmarts(motif)))
    return fp


4) Apply your function to the dataframe and store your fingerprints in a new column.

In [65]:
df_ecoli["my_fp"] = df_ecoli["molecule"].apply(substructures)
df_delaney["my_fp"] = df_delaney["molecule"].apply(substructures)

In [66]:
df_ecoli.head()

,chembl_id,smiles,mic,unit,molecule,my_fp,morgan_fp
0,CHEMBL1689756,CN(C)[C@@H]1C(O)=C(C(N)=O)C(=O)[C@@]2(O)C(O)=C...,0.25,ug.mL-1,<rdkit.Chem.rdchem.Mol object at 0x14a870190>,"[False, False, True, True, False, True, True, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CHEMBL1222951,CO/N=C1/CCN(c2c(F)cc3c(=O)c(C(=O)O)cn(C4CC4)c3...,8.00,ug.mL-1,<rdkit.Chem.rdchem.Mol object at 0x14a870270>,"[True, True, False, True, False, True, True, T...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CHEMBL2403434,COc1cccc(CC(=O)Nc2nc(-c3ccc(Br)cc3)cs2)c1,12.50,ug.mL-1,<rdkit.Chem.rdchem.Mol object at 0x14a870350>,"[False, True, True, False, False, False, False...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,CHEMBL148,C[C@@H](O)[C@H]1C(=O)N2C(C(=O)O)=C(SCCNC=N)C[C...,0.19,ug.mL-1,<rdkit.Chem.rdchem.Mol object at 0x14a870430>,"[True, False, True, False, False, True, True, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,CHEMBL22,COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC,0.25,ug.mL-1,<rdkit.Chem.rdchem.Mol object at 0x14a870510>,"[False, True, False, False, False, False, True...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [56]:
df_delaney.head()

,Compound ID,logS,smiles,molecule,my_fp
0,"1,1,1,2-Tetrachloroethane",-2.18,ClCC(Cl)(Cl)Cl,<rdkit.Chem.rdchem.Mol object at 0x147f045f0>,"[False, False, False, False, False, False, Fal..."
1,"1,1,1-Trichloroethane",-2.00,CC(Cl)(Cl)Cl,<rdkit.Chem.rdchem.Mol object at 0x147f046d0>,"[False, False, False, False, False, False, Fal..."
2,"1,1,2,2-Tetrachloroethane",-1.74,ClC(Cl)C(Cl)Cl,<rdkit.Chem.rdchem.Mol object at 0x147f047b0>,"[False, False, False, False, False, False, Fal..."
3,"1,1,2-Trichloroethane",-1.48,ClCC(Cl)Cl,<rdkit.Chem.rdchem.Mol object at 0x147f04890>,"[False, False, False, False, False, False, Fal..."
4,"1,1,2-Trichlorotrifluoroethane",-3.04,FC(F)(Cl)C(F)(Cl)Cl,<rdkit.Chem.rdchem.Mol object at 0x147f04970>,"[False, False, False, False, False, False, Fal..."


5) Create Morgan fingerprints as spearate feature (e.g. as column in your df).

In [62]:
from rdkit.Chem import rdFingerprintGenerator, DataStructs
def mol2fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048)
    fp = mfpgen.GetFingerprint(mol)
    return fp

df_delaney["morgan_fp"] = df_delaney["smiles"].apply(mol2fp)
df_ecoli["morgan_fp"] = df_ecoli["smiles"].apply(mol2fp)

6) Calculate all physical descriptors available in RDKit and add them to a new dataframe (hint: `Descriptors.CalcMolDescriptors` returns a dictionary of all available descriptors)

In [70]:
from rdkit import Chem
from rdkit.Chem import Descriptors

In [100]:
def calc_descriptors(molecule):
    desc = Descriptors.CalcMolDescriptors(molecule)
    return desc

phys_desc_ecoli = pd.DataFrame([calc_descriptors(m) for m in df_ecoli["molecule"]])
phys_desc_delaney = pd.DataFrame([calc_descriptors(m) for m in df_delaney["molecule"]])

In [102]:
phys_desc_ecoli.head()

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,13.299498,13.299498,0.034495,-2.691949,0.250126,29.718750,464.862,443.694,464.109877,170,...,0,0,0,0,0,0,0,0,0,0
1,15.611530,15.611530,0.042858,-1.384387,0.502987,22.090909,458.490,431.274,458.196548,176,...,0,0,0,0,0,0,0,0,0,0
2,12.180454,12.180454,0.097260,-0.097260,0.668333,10.416667,403.301,388.181,402.003761,122,...,0,0,0,0,0,1,0,0,0,0
3,11.938415,11.938415,0.052549,-1.104254,0.224452,26.100000,299.352,282.216,299.093977,110,...,1,0,0,0,0,0,0,0,0,0
4,5.848108,5.848108,0.153835,0.153835,0.853356,10.238095,290.323,272.179,290.137890,112,...,0,0,0,0,0,0,0,0,0,0


In [101]:
phys_desc_delaney.head()

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,5.116512,5.116512,0.039352,-1.276235,0.487138,12.00,167.850,165.834,165.891061,38,...,0,0,0,0,0,0,0,0,0,0
1,5.060957,5.060957,1.083333,-1.083333,0.445171,12.00,133.405,130.381,131.930033,32,...,0,0,0,0,0,0,0,0,0,0
2,5.114198,5.114198,0.672840,-0.672840,0.527312,11.00,167.850,165.834,165.891061,38,...,0,0,0,0,0,0,0,0,0,0
3,5.095679,5.095679,0.308642,-0.405864,0.480258,9.60,133.405,130.381,131.930033,32,...,0,0,0,0,0,0,0,0,0,0
4,11.544753,11.544753,3.685957,-4.226080,0.553756,14.25,187.375,187.375,185.901768,50,...,0,0,0,0,0,0,0,0,0,0


7) Use a Random Forest Regressor to predict the target to compare:
- your custom fingerprint,
- the Morgan fingerprints,
- the physical descriptors.
Which one would be the best feature set to predict your target? Note that for this task, the fingerprints should be represented as matrix, not one single entry like in the dataframe (snippet provided below). Interpret the results!

In [113]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
def prediction(target, df, descr_df):
    y = target
    res=[]
    for i in ["my_fp", "morgan_fp", "phys"]:
        if i =="phys":
            X = descr_df  
        else: X = np.vstack(df[i])  # for descriptors, you can simply use X = descr_df            
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        rf = RandomForestRegressor(n_estimators=200, random_state=42)
        rf.fit(X_train, y_train)

        pred_train = rf.predict(X_train)
        pred_test = rf.predict(X_test)
        dic = {"Fingerprint": i, "Train R2": r2_score(y_train, pred_train), "Test  R2": r2_score(y_test, pred_test), 
               "Test RMSE": mean_squared_error(y_test, pred_test) ** 0.5, "Test MAE" :mean_absolute_error(y_test, pred_test)}
        res.append(dic)
        
    res_df = pd.DataFrame(res)
    return res_df

In [114]:
res_delaney = prediction(df_delaney["logS"], df_delaney, phys_desc_delaney)

In [115]:
res_ecoli = prediction(df_ecoli["mic"], df_ecoli, phys_desc_ecoli)

In [116]:
res_delaney.head()

,Fingerprint,Train R2,Test R2,Test RMSE,Test MAE
0,my_fp,0.317505,0.296572,1.750173,1.361049
1,morgan_fp,0.942902,0.699283,1.144327,0.857881
2,phys,0.986531,0.909296,0.628469,0.460560


In [117]:
res_ecoli.head()

,Fingerprint,Train R2,Test R2,Test RMSE,Test MAE
0,my_fp,0.036374,-0.014623,6968.421138,746.904545
1,morgan_fp,0.842520,0.142002,6408.039746,1350.621783
2,phys,0.863280,-0.012459,6960.985887,723.123139


8) Compare the results for the two datasets and discuss!